# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-2/ML-week-1-FLY/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Scoring / Ranking.**

My lane (Lane 4: CTR/Engagement Opportunity Scoring, from w01) is fundamentally about producing an ordered list — which visible pages should an editor look at first — not a single yes/no label. That's ranking, per the task-type table: "which ones first?" maps to ranking/scoring with a priority score and precision@K as the typical metric.

Underneath the ranking, there's a simpler binary idea I can also frame as classification if it's useful later: "is this page a real CTR-gap outlier for its position tier?" (yes/no). But the actual deliverable an editor uses is the ranked list, not an isolated flag, so I'm treating ranking as the primary task type and the binary version as a supporting building block.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**Proxy, not an observed future outcome — and I want to be honest about that.**

The score I'd rank pages by is `ctr_gap = tier_median_ctr - page_ctr`: how far a page's CTR sits below the median CTR of other pages at the same position tier, computed entirely from this window's `impressions_90d`, `clicks_90d` (via `ctr`), and `avg_position` — all observed signals, never a product decision flag like `health_score`. That keeps it leakage-safe by construction, since there's no rule-output column to accidentally copy.

But it IS a proxy, not a true future outcome: it tells me a page is underperforming its peers *right now*, not that fixing its title will actually recover clicks *later*. A stronger version of this target (for a future week, once I bring in the warehouse's daily table) would be a forward-looking one — does CTR improve in the 30 days after a page gets flagged and reviewed. For this week's framing, the current-window gap is the honest starting proxy.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Precision@50**, checked against a stricter independent bar: of the top 50 pages my ranking puts first (ranked by `ctr_gap`), what fraction also clear a stricter, separately-defined threshold (CTR at least 30% below their tier's median, not just any amount below)?

I'm using a stricter *second* rule as the check (rather than checking the ranking against itself) so it's not circular — the ranking and the pass/fail bar are computed independently, even though both come from the same observed columns. Precision@50 fits because editor capacity is the real constraint here (per the lane guide: "Precision@50 if the team can act on 50 candidates") — I care about correctness in the top of the list, not overall accuracy across all 12,000 visible pages.

I compare it against a random-order baseline of 50 pages from the same visible pool, so "good" has a number to beat, not just a vibe.


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
visible["tier_median_ctr"] = visible["position_tier"].map(tier_median_ctr)
visible["ctr_gap"] = visible["tier_median_ctr"] - visible["ctr"]
visible["ctr_gap_pct_of_median"] = visible["ctr_gap"] / visible["tier_median_ctr"]

# Rank by gap, take the top 50 an editor with limited time would actually review
ranked = visible.sort_values("ctr_gap", ascending=False)
top50 = ranked.head(50)

# Independent stricter bar: CTR at least 30% below tier median (not the same rule used to rank)
precision_at_50 = (top50["ctr_gap_pct_of_median"] >= 0.30).mean()
print(f"Precision@50 (ranked top 50 vs. independent >=30% underperform bar): {precision_at_50:.3f} "
      f"(~{round(precision_at_50*50)} of 50)")

# Random-order baseline for comparison
rand_sample = visible.sample(50, random_state=42)
rand_precision = (rand_sample["ctr_gap_pct_of_median"] >= 0.30).mean()
print(f"Same check on a random sample of 50 pages (baseline): {rand_precision:.3f} "
      f"(~{round(rand_precision*50)} of 50)")


Precision@50 (ranked top 50 vs. independent >=30% underperform bar): 1.000 (~50 of 50)
Same check on a random sample of 50 pages (baseline): 0.340 (~17 of 50)


## 4. The unit of analysis, as a real dataframe

**One row = one visible content page** (`content_id`), restricted to pages with enough exposure to judge fairly (`impressions_90d >= 500`) and a real, known position (`avg_position` between 1 and 20 — excluding the `avg_position == 0` rows, which the data dictionary flags as "no data," not rank zero). `client_id` and `content_id` are shown only as pseudonymized join/grouping keys, never as features.


In [4]:
print(f"Shape: {visible.shape[0]} rows x {visible.shape[1]} columns\n")

cols_to_show = ["content_id", "client_id", "position_tier", "avg_position",
                "impressions_90d", "ctr", "tier_median_ctr", "ctr_gap"]
visible[cols_to_show].sort_values("ctr_gap", ascending=False).head(8)


Shape: 12023 rows x 47 columns



,content_id,client_id,position_tier,avg_position,impressions_90d,ctr,tier_median_ctr,ctr_gap
19476,content_adacaaec0453,client_e629fa6598,page_1,4.6,905,0.0,0.24,0.24
29886,content_04d69956e256,client_19581e27de,page_1,3.7,742,0.0,0.24,0.24
19482,content_018cf0021b5d,client_6208ef0f77,page_1,7.3,1174,0.0,0.24,0.24
183,content_ce611830d125,client_19581e27de,page_1,6.9,1409,0.0,0.24,0.24
171,content_214a94adfa00,client_19581e27de,page_1,6.9,1168,0.0,0.24,0.24
12398,content_73a9cfafddca,client_19581e27de,page_1,5.3,726,0.0,0.24,0.24
29983,content_6880eb215048,client_19581e27de,page_1,6.8,2845,0.0,0.24,0.24
19363,content_85925d6c92b8,client_f369cb89fc,page_1,9.6,2049,0.0,0.24,0.24


## 5. Why ML beats a fixed rule here

A single global "flag pages with CTR under X%" rule genuinely doesn't work — the numbers above show why: median CTR is roughly 0.24% at `page_1` but only 0.155% at `page_3_5`, so one flat cutoff would either miss real problems at good positions or falsely flag totally normal pages at weaker positions. That's already reason enough to need *some* structure smarter than an if-statement — at minimum, a per-tier comparison instead of a global one.

Where this grows past "just write a slightly smarter if-statement" and into a genuine scoring/ranking problem: position tier is only one of several signals that plausibly affect what "expected" CTR should look like for a page — content type, search intent, freshness, and word count could all shift the fair baseline too. Hand-writing a nested if/else for every combination of tier × content_type × intent gets unwieldy fast, and each new signal doubles the branches. A scoring approach (weighting multiple observed signals, learned or tuned against a validation check) generalizes across those combinations without me manually enumerating every case — and it's the same reasoning the starter pipeline itself demonstrated in w01/w02: a rule with one threshold (baseline, Precision@50 = 0.240) got clearly beaten by a model that could weigh several signals together (random forest, Precision@50 = 0.740).


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.